In [1]:
from langgraph.func import entrypoint
from langgraph.store.postgres import AsyncPostgresStore
from langchain_core.runnables import RunnableConfig
from langgraph.store.memory import InMemoryStore
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langmem import create_memory_store_manager, ReflectionExecutor

/home/frank_shan/miniconda3/envs/pyapi/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedding_model = OllamaEmbeddings(model="qwen3-embedding:8b")
chat_model = ChatOllama(model="qwen3:8b", temperature=0)

dummy_vec = embedding_model.embed_query("test")
EMBEDDING_DIM = len(dummy_vec)
print(f"Detected embedding dimension: {EMBEDDING_DIM}")

Detected embedding dimension: 4096


In [5]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
store = InMemoryStore(
    index={
        "dims": 4096,
        "embed": embedding_model,
    }
)

memory_manager = create_memory_store_manager(
    chat_model,
    namespace=("memories", "{user_id}")
)

@entrypoint(store=store)
async def my_agent(message: str, config: RunnableConfig):
    memories = await memory_manager.asearch(
        query=message,
        config=config,
    )
    llm_response = chat_model.invoke([HumanMessage(content=message), SystemMessage(content=f"## Memories from the user:\n{str(memories)}")])
    response = {"role": "assistant", "content": llm_response.content}

    await memory_manager.ainvoke(
        {"messages": [{"role": "user", "content": message}, response]},
    )
    return response

config = {"configurable": {"user_id": "user123"}}
response_1 = await my_agent.ainvoke(
    "I prefer dark mode in all my apps",
    config=config,
    stream_mode="values"
)
print("response_1:", response_1)
# Later conversation - automatically retrieves and uses the stored preference
response_2 = await my_agent.ainvoke(
    "Actually, I prefer light mode in all my apps",
    config=config,
    stream_mode="values"
)
print("response_2:", response_2)
response_3 = await my_agent.ainvoke(
    "What theme do I prefer?",
    config=config,
    stream_mode="values"
)
print("response_3:", response_3)
# You can list over memories in the user's namespace manually:
print(memory_manager.search(query="app preferences", config=config))


response_1: {'role': 'assistant', 'content': "I understand that dark mode is a popular choice for many users! It can be great for reducing eye strain in low-light environments, saving battery life on OLED screens, or simply personal preference. If you're looking for tips, recommendations, or help with specific apps, feel free to share — I'd be happy to assist! 🌙"}
response_2: {'role': 'assistant', 'content': "Apologies for the confusion earlier! 🌞 It sounds like you're more comfortable with light mode—maybe it's better for your workflow or reduces eye strain during the day? If you'd like tips for optimizing light mode (e.g., adjusting brightness, reducing blue light), or if there's anything else I can assist with, feel free to let me know! 😊"}
response_3: {'role': 'assistant', 'content': 'Based on your preferences, you favor **light mode** across all applications. This is a common choice, and the benefits include reduced eye strain, especially in low-light environments, and improved ba